In [1]:
#Install required libraries
!pip install -q datasets pandas langdetect huggingface_hub

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
#Import libraries and project configuration

import pandas as pd
import re
from collections import Counter
from datasets import load_dataset
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

# Make language detection repeatable
DetectorFactory.seed = 42

# Dataset configuration
DATASET_ID = "AmazonScience/massive"

LANGUAGES = {
    "en-US": "English",
    "hi-IN": "Hindi",
    "es-ES": "Spanish",
    "te-IN": "Telugu"
}

RECORDS_PER_LANGUAGE = 4
SPLIT = "train"

# Language code mapping
LANG_CODE_MAP = {
    "en": "English",
    "hi": "Hindi",
    "es": "Spanish",
    "te": "Telugu"
}

print("Project configuration completed!")
print("Dataset:", DATASET_ID)
print("Languages:", list(LANGUAGES.values()))
print("Records per language:", RECORDS_PER_LANGUAGE)

Project configuration completed!
Dataset: AmazonScience/massive
Languages: ['English', 'Hindi', 'Spanish', 'Telugu']
Records per language: 4


In [3]:

!pip install -q huggingface_hub pandas pyarrow datasets

import pandas as pd
from datasets import Dataset
from huggingface_hub import list_repo_files, hf_hub_download

DATASET_ID = "AmazonScience/massive"
DATASET_REVISION = "refs/convert/parquet"

languages = {
    "English": "en-US",
    "Hindi": "hi-IN",
    "Spanish": "es-ES",
    "Telugu": "te-IN"
}

datasets_by_language = {}
loading_errors = {}

# Get files from the Parquet conversion branch
all_files = list_repo_files(
    repo_id=DATASET_ID,
    repo_type="dataset",
    revision=DATASET_REVISION
)

print("Total Parquet branch files:", len(all_files))

for language, locale in languages.items():

    try:
        # Find the training file for this language
        matching_files = [
            f for f in all_files
            if f.startswith(locale + "/")
            and f.endswith(".parquet")
            and "train" in f.lower()
        ]

        if not matching_files:
            raise FileNotFoundError(
                f"No training Parquet file found for {locale}"
            )

        file_path = matching_files[0]

        print(f"\nLoading {language}: {file_path}")

        local_file = hf_hub_download(
            repo_id=DATASET_ID,
            filename=file_path,
            repo_type="dataset",
            revision=DATASET_REVISION
        )

        df = pd.read_parquet(local_file)

        datasets_by_language[language] = Dataset.from_pandas(
            df,
            preserve_index=False
        )

        print(f"{language} loaded successfully!")
        print("Records:", len(df))
        print("Columns:", list(df.columns))

    except Exception as e:
        loading_errors[language] = str(e)
        print(f"{language} failed:", e)

print("\n========== DATASET LOADING SUMMARY ==========")
print("Successfully loaded:", len(datasets_by_language))
print("Failed:", len(loading_errors))

if not datasets_by_language:
    print("\nDetailed errors:")
    for language, error in loading_errors.items():
        print(language, ":", error)
    raise RuntimeError("Dataset loading failed.")

print("\nDataset loading completed successfully!")


Total Parquet branch files: 163

Loading English: en-US/train/0000.parquet


0000.parquet:   0%|          | 0.00/676k [00:00<?, ?B/s]

English loaded successfully!
Records: 11514
Columns: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments']

Loading Hindi: hi-IN/train/0000.parquet


hi-IN/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 1.10MB            

hi-IN/train/0000.parquet: downloading bytes:           |  0.00B            

Hindi loaded successfully!
Records: 11514
Columns: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments']

Loading Spanish: es-ES/train/0000.parquet


0000.parquet:   0%|          | 0.00/806k [00:00<?, ?B/s]

Spanish loaded successfully!
Records: 11514
Columns: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments']

Loading Telugu: te-IN/train/0000.parquet


te-IN/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 1.18MB            

te-IN/train/0000.parquet: downloading bytes:           |  0.00B            

Telugu loaded successfully!
Records: 11514
Columns: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments']

========== DATASET LOADING SUMMARY ==========
Successfully loaded: 4
Failed: 0

Dataset loading completed successfully!


In [4]:

# STEP 4: COMBINE ALL FOUR LANGUAGES

import pandas as pd

dataframes = []

for language, dataset in datasets_by_language.items():

    temp_df = dataset.to_pandas()
    temp_df["language_name"] = language

    dataframes.append(temp_df)

df = pd.concat(dataframes, ignore_index=True)

print("Total records:", len(df))
print("\nDataset shape:", df.shape)

print("\nLanguage distribution:")
print(df["language_name"].value_counts())

print("\nAvailable columns:")
print(df.columns.tolist())

display(df[[
    "language_name",
    "locale",
    "utt",
    "intent",
    "scenario"
]].head(10))


Total records: 46056

Dataset shape: (46056, 11)

Language distribution:
language_name
English    11514
Hindi      11514
Spanish    11514
Telugu     11514
Name: count, dtype: int64

Available columns:
['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments', 'language_name']


,language_name,locale,utt,intent,scenario
0,English,en-US,wake me up at nine am on friday,48,16
1,English,en-US,set an alarm for two hours from now,48,16
2,English,en-US,olly quiet,46,10
3,English,en-US,stop,46,10
4,English,en-US,olly pause for ten seconds,46,10
5,English,en-US,pause for ten seconds,46,10
6,English,en-US,make the lighting bit more warm here,1,8
7,English,en-US,please set the lighting suitable for reading,1,8
8,English,en-US,time to sleep,40,8
9,English,en-US,time to sleep olly,40,8


In [5]:

# STEP 5: INSPECT REAL DATASET RECORDS

print("Sample customer messages:\n")

for language in ["English", "Hindi", "Spanish", "Telugu"]:

    sample = df[df["language_name"] == language].iloc[0]

    print("\nLanguage:", language)
    print("Customer message:", sample["utt"])
    print("Intent:", sample["intent"])
    print("Scenario:", sample["scenario"])
    print("Annotated message:", sample["annot_utt"])
    print("Judgments:", sample["judgments"])

print("\nColumn data types:")
print(df.dtypes)


Sample customer messages:


Language: English
Customer message: wake me up at nine am on friday
Intent: 48
Scenario: 16
Annotated message: wake me up at [time : nine am] on [date : friday]
Judgments: {'grammar_score': array([], dtype=int8), 'intent_score': array([], dtype=int8), 'language_identification': array([], dtype=object), 'slots_score': array([], dtype=int8), 'spelling_score': array([], dtype=int8), 'worker_id': array([], dtype=object)}

Language: Hindi
Customer message: शुक्रवार को सुबह नौ बजे मुझे जगा दो
Intent: 48
Scenario: 16
Annotated message: [date : शुक्रवार] को सुबह [time : नौ बजे] मुझे जगा दो
Judgments: {'grammar_score': array([4, 3, 3], dtype=int8), 'intent_score': array([1, 1, 1], dtype=int8), 'language_identification': array(['target', 'target', 'target'], dtype=object), 'slots_score': array([1, 1, 1], dtype=int8), 'spelling_score': array([2, 2, 2], dtype=int8), 'worker_id': array(['16', '3', '42'], dtype=object)}

Language: Spanish
Customer message: despiértame a l

In [6]:

# STEP 6: INSTALL AND PREPARE LANGUAGE DETECTION

!pip install -q langdetect

from langdetect import detect_langs, DetectorFactory

# Make detection results reproducible
DetectorFactory.seed = 42

print("Language detection library is ready.")


Language detection library is ready.


In [7]:
#Detect the customer's language

# STEP 7: LANGUAGE DETECTION FUNCTION

def detect_language(text):

    if not isinstance(text, str) or not text.strip():
        return "Unknown"

    # Detect Hindi script
    if any("\u0900" <= char <= "\u097F" for char in text):
        return "Hindi"

    # Detect Telugu script
    if any("\u0C00" <= char <= "\u0C7F" for char in text):
        return "Telugu"

    # Detect English or Spanish
    try:
        predictions = detect_langs(text)
        best_prediction = predictions[0]

        if best_prediction.prob < 0.70:
            return "Unknown"

        if best_prediction.lang == "en":
            return "English"

        if best_prediction.lang == "es":
            return "Spanish"

        return "Unknown"

    except:
        return "Unknown"


# Apply detection to all dataset messages
df["detected_language"] = df["utt"].apply(detect_language)

print("Language detection completed!")

print("\nDetected language counts:")
print(df["detected_language"].value_counts())


Language detection completed!

Detected language counts:
detected_language
Telugu     11514
Hindi      11499
English     9974
Spanish     9767
Unknown     3302
Name: count, dtype: int64


In [10]:
print("Available columns:")
print(df.columns.tolist())
language_mapping = {
    "en-US": "English",
    "hi-IN": "Hindi",
    "es-ES": "Spanish",
    "te-IN": "Telugu"
}

df["actual_language"] = df["locale"].map(language_mapping)

print("Actual language column created!")
print(df["actual_language"].value_counts())

Available columns:
['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments', 'language_name', 'detected_language', 'issue_category']
Actual language column created!
actual_language
English    11514
Hindi      11514
Spanish    11514
Telugu     11514
Name: count, dtype: int64


In [11]:
# STEP 8: CUSTOMER ISSUE IDENTIFICATION

def identify_issue(message):

    message = str(message).lower()

    # Network / Internet problems
    network_words = [
        "network", "internet", "wifi", "wi-fi",
        "connection", "connect", "offline",
        "नेटवर्क", "इंटरनेट", "वाईफाई",
        "red", "internet no funciona", "sin internet",
        "conexión", "wifi no funciona",
        "నెట్‌వర్క్", "ఇంటర్నెట్", "వైఫై",
        "కనెక్షన్", "పనిచేయడం లేదు"
    ]

    # Password problems
    password_words = [
        "password", "forgot password", "login",
        "cannot login", "can't login",
        "contraseña", "iniciar sesión",
        "पासवर्ड", "लॉगिन",
        "పాస్‌వర్డ్", "లాగిన్"
    ]

    # Billing problems
    billing_words = [
        "bill", "billing", "payment", "charged",
        "invoice", "refund",
        "factura", "pago", "reembolso",
        "बिल", "भुगतान",
        "బిల్లు", "చెల్లింపు"
    ]

    # Slow service problems
    slow_words = [
        "slow", "very slow", "speed",
        "not loading", "buffering",
        "lento", "velocidad",
        "धीमा", "स्पीड",
        "నెమ్మదిగా", "స్పీడ్"
    ]

    if any(word in message for word in network_words):
        return "Network Issue"

    elif any(word in message for word in password_words):
        return "Password Issue"

    elif any(word in message for word in billing_words):
        return "Billing Issue"

    elif any(word in message for word in slow_words):
        return "Slow Service"

    else:
        return "Other Issue"


# Apply issue identification to the dataset
df["issue_category"] = df["utt"].apply(identify_issue)

print("Issue identification completed!")

print("\nIssue distribution:")
print(df["issue_category"].value_counts())

display(df[
    [
        "utt",
        "locale",
        "actual_language",
        "language_name",
        "detected_language",
        "issue_category"
    ]
].head(10))

Issue identification completed!

Issue distribution:
issue_category
Other Issue       45457
Network Issue       316
Billing Issue       250
Slow Service         27
Password Issue        6
Name: count, dtype: int64


,utt,locale,actual_language,language_name,detected_language,issue_category
0,wake me up at nine am on friday,en-US,English,English,English,Other Issue
1,set an alarm for two hours from now,en-US,English,English,English,Other Issue
2,olly quiet,en-US,English,English,Unknown,Other Issue
3,stop,en-US,English,English,Unknown,Other Issue
4,olly pause for ten seconds,en-US,English,English,English,Other Issue
5,pause for ten seconds,en-US,English,English,English,Other Issue
6,make the lighting bit more warm here,en-US,English,English,English,Other Issue
7,please set the lighting suitable for reading,en-US,English,English,English,Other Issue
8,time to sleep,en-US,English,English,English,Other Issue
9,time to sleep olly,en-US,English,English,Unknown,Other Issue


In [12]:
#Generate relevant troubleshooting responses

# STEP 9: MULTILINGUAL TROUBLESHOOTING RESPONSES

responses = {

    "Network Issue": {
        "English": (
            "We are sorry that your network is not working. "
            "Please restart your router, check the cable connections, "
            "and reconnect your device to Wi-Fi. "
            "If the problem continues, please contact our support team."
        ),

        "Hindi": (
            "आपका नेटवर्क काम नहीं कर रहा है, इसके लिए हमें खेद है। "
            "कृपया अपना राउटर रीस्टार्ट करें, केबल कनेक्शन जांचें "
            "और अपने डिवाइस को फिर से वाई-फाई से जोड़ें। "
            "समस्या जारी रहने पर हमारी सहायता टीम से संपर्क करें।"
        ),

        "Spanish": (
            "Lamentamos que su red no esté funcionando. "
            "Reinicie su router, revise los cables y vuelva a conectar "
            "su dispositivo al Wi-Fi. Si el problema continúa, "
            "contacte con nuestro equipo de soporte."
        ),

        "Telugu": (
            "మీ నెట్‌వర్క్ పనిచేయకపోవడం పట్ల మేము చింతిస్తున్నాము. "
            "దయచేసి మీ రౌటర్‌ను రీస్టార్ట్ చేయండి, కేబుల్ కనెక్షన్లను "
            "తనిఖీ చేయండి మరియు మీ పరికరాన్ని మళ్లీ వై-ఫైకు కనెక్ట్ చేయండి. "
            "సమస్య కొనసాగితే మా సహాయ బృందాన్ని సంప్రదించండి."
        )
    },

    "Password Issue": {
        "English": (
            "We can help you with your login problem. "
            "Please check your username and use the Forgot Password "
            "option to reset your password."
        ),

        "Hindi": (
            "हम आपकी लॉगिन समस्या में मदद कर सकते हैं। "
            "कृपया अपना उपयोगकर्ता नाम जांचें और पासवर्ड रीसेट करने "
            "के लिए Forgot Password विकल्प का उपयोग करें।"
        ),

        "Spanish": (
            "Podemos ayudarle con su problema de inicio de sesión. "
            "Verifique su nombre de usuario y utilice la opción "
            "Olvidé mi contraseña para restablecerla."
        ),

        "Telugu": (
            "మీ లాగిన్ సమస్యలో మేము సహాయం చేస్తాము. "
            "దయచేసి మీ వినియోగదారు పేరును తనిఖీ చేసి, "
            "పాస్‌వర్డ్‌ను రీసెట్ చేయడానికి Forgot Password ఎంపికను ఉపయోగించండి."
        )
    },

    "Billing Issue": {
        "English": (
            "We can help you with your billing concern. "
            "Please check your latest invoice and payment status. "
            "If you notice an incorrect charge, contact our billing team."
        ),

        "Hindi": (
            "हम आपकी बिलिंग समस्या में मदद कर सकते हैं। "
            "कृपया अपना नवीनतम बिल और भुगतान की स्थिति जांचें। "
            "गलत शुल्क दिखाई देने पर हमारी बिलिंग टीम से संपर्क करें।"
        ),

        "Spanish": (
            "Podemos ayudarle con su problema de facturación. "
            "Revise su última factura y el estado del pago. "
            "Si encuentra un cargo incorrecto, contacte con nuestro equipo."
        ),

        "Telugu": (
            "మీ బిల్లింగ్ సమస్యలో మేము సహాయం చేస్తాము. "
            "దయచేసి మీ తాజా బిల్లు మరియు చెల్లింపు స్థితిని తనిఖీ చేయండి. "
            "తప్పు ఛార్జీ కనిపిస్తే మా బిల్లింగ్ బృందాన్ని సంప్రదించండి."
        )
    },

    "Slow Service": {
        "English": (
            "We are sorry that the service is slow. "
            "Please restart your device and router, disconnect "
            "unused devices, and try again."
        ),

        "Hindi": (
            "सेवा धीमी होने के लिए हमें खेद है। "
            "कृपया अपना डिवाइस और राउटर रीस्टार्ट करें, "
            "अनावश्यक डिवाइस डिस्कनेक्ट करें और फिर से प्रयास करें।"
        ),

        "Spanish": (
            "Lamentamos que el servicio esté lento. "
            "Reinicie su dispositivo y router, desconecte los "
            "dispositivos que no utiliza e inténtelo de nuevo."
        ),

        "Telugu": (
            "సేవ నెమ్మదిగా ఉన్నందుకు మేము చింతిస్తున్నాము. "
            "దయచేసి మీ పరికరం మరియు రౌటర్‌ను రీస్టార్ట్ చేసి, "
            "అవసరం లేని పరికరాలను డిస్‌కనెక్ట్ చేసి మళ్లీ ప్రయత్నించండి."
        )
    },

    "Other Issue": {
        "English": (
            "Thank you for contacting support. "
            "Could you please explain your problem in more detail? "
            "Our support team will help you."
        ),

        "Hindi": (
            "सहायता टीम से संपर्क करने के लिए धन्यवाद। "
            "कृपया अपनी समस्या के बारे में अधिक जानकारी दें। "
            "हमारी टीम आपकी मदद करेगी।"
        ),

        "Spanish": (
            "Gracias por contactar con soporte. "
            "¿Podría explicar su problema con más detalle? "
            "Nuestro equipo le ayudará."
        ),

        "Telugu": (
            "మా సహాయ బృందాన్ని సంప్రదించినందుకు ధన్యవాదాలు. "
            "దయచేసి మీ సమస్యను మరింత వివరంగా తెలియజేయండి. "
            "మా బృందం మీకు సహాయం చేస్తుంది."
        )
    }
}


def generate_support_reply(issue, language):

    return responses.get(
        issue,
        responses["Other Issue"]
    ).get(
        language,
        responses["Other Issue"]["English"]
    )


# Generate replies for all dataset messages
df["customer_reply"] = df.apply(
    lambda row: generate_support_reply(
        row["issue_category"],
        row["detected_language"]
    ),
    axis=1
)

print("Troubleshooting replies generated!")

display(df[[
    "utt",
    "detected_language",
    "issue_category",
    "customer_reply"
]].head(15))


Troubleshooting replies generated!


,utt,detected_language,issue_category,customer_reply
0,wake me up at nine am on friday,English,Other Issue,Thank you for contacting support. Could you pl...
1,set an alarm for two hours from now,English,Other Issue,Thank you for contacting support. Could you pl...
2,olly quiet,Unknown,Other Issue,Thank you for contacting support. Could you pl...
3,stop,Unknown,Other Issue,Thank you for contacting support. Could you pl...
4,olly pause for ten seconds,English,Other Issue,Thank you for contacting support. Could you pl...
5,pause for ten seconds,English,Other Issue,Thank you for contacting support. Could you pl...
6,make the lighting bit more warm here,English,Other Issue,Thank you for contacting support. Could you pl...
7,please set the lighting suitable for reading,English,Other Issue,Thank you for contacting support. Could you pl...
8,time to sleep,English,Other Issue,Thank you for contacting support. Could you pl...
9,time to sleep olly,Unknown,Other Issue,Thank you for contacting support. Could you pl...


In [13]:
#Create English internal support logs

# STEP 10: INTERNAL SUPPORT LOGS IN ENGLISH

def create_internal_log(row):

    return (
        f"Customer language: {row['detected_language']}. "
        f"Issue category: {row['issue_category']}. "
        f"Original message: {row['utt']}. "
        f"Recommended response: {row['customer_reply']}. "
        f"Status: Automated response generated; "
        f"awaiting customer confirmation."
    )


df["internal_english_log"] = df.apply(
    create_internal_log,
    axis=1
)

print("English internal support logs created!")

display(df[[
    "utt",
    "issue_category",
    "customer_reply",
    "internal_english_log"
]].head(10))


English internal support logs created!


,utt,issue_category,customer_reply,internal_english_log
0,wake me up at nine am on friday,Other Issue,Thank you for contacting support. Could you pl...,Customer language: English. Issue category: Ot...
1,set an alarm for two hours from now,Other Issue,Thank you for contacting support. Could you pl...,Customer language: English. Issue category: Ot...
2,olly quiet,Other Issue,Thank you for contacting support. Could you pl...,Customer language: Unknown. Issue category: Ot...
3,stop,Other Issue,Thank you for contacting support. Could you pl...,Customer language: Unknown. Issue category: Ot...
4,olly pause for ten seconds,Other Issue,Thank you for contacting support. Could you pl...,Customer language: English. Issue category: Ot...
5,pause for ten seconds,Other Issue,Thank you for contacting support. Could you pl...,Customer language: English. Issue category: Ot...
6,make the lighting bit more warm here,Other Issue,Thank you for contacting support. Could you pl...,Customer language: English. Issue category: Ot...
7,please set the lighting suitable for reading,Other Issue,Thank you for contacting support. Could you pl...,Customer language: English. Issue category: Ot...
8,time to sleep,Other Issue,Thank you for contacting support. Could you pl...,Customer language: English. Issue category: Ot...
9,time to sleep olly,Other Issue,Thank you for contacting support. Could you pl...,Customer language: Unknown. Issue category: Ot...


In [14]:
# STEP 11: TEST THE SUPPORT DESK WITH NEW MESSAGES
def support_desk(message):
    language = detect_language(message)
    issue = identify_issue(message)
    reply = generate_support_reply(
        issue,
        language
    )
    internal_log = (
        f"Customer language: {language}. "
        f"Issue: {issue}. "
        f"Original message: {message}. "
        f"Recommended response: {reply}. "
        f"Status: Awaiting customer confirmation."
    )
    print("CUSTOMER MESSAGE:")
    print(message)
    print("\nDETECTED LANGUAGE:")
    print(language)
    print("\nIDENTIFIED ISSUE:")
    print(issue)
    print("\nCUSTOMER SUPPORT REPLY:")
    print(reply)
    print("\nINTERNAL LOG (ENGLISH):")
    print(internal_log)
# Test with a network problem
#support_desk("My network is not working. I cannot connect to the internet.")
#support_desk("Olvidé mi contraseña y no puedo iniciar sesión")
support_desk("నా ఇంటర్నెట్ పనిచేయడం లేదు")

CUSTOMER MESSAGE:
నా ఇంటర్నెట్ పనిచేయడం లేదు

DETECTED LANGUAGE:
Telugu

IDENTIFIED ISSUE:
Network Issue

CUSTOMER SUPPORT REPLY:
మీ నెట్‌వర్క్ పనిచేయకపోవడం పట్ల మేము చింతిస్తున్నాము. దయచేసి మీ రౌటర్‌ను రీస్టార్ట్ చేయండి, కేబుల్ కనెక్షన్లను తనిఖీ చేయండి మరియు మీ పరికరాన్ని మళ్లీ వై-ఫైకు కనెక్ట్ చేయండి. సమస్య కొనసాగితే మా సహాయ బృందాన్ని సంప్రదించండి.

INTERNAL LOG (ENGLISH):
Customer language: Telugu. Issue: Network Issue. Original message: నా ఇంటర్నెట్ పనిచేయడం లేదు. Recommended response: మీ నెట్‌వర్క్ పనిచేయకపోవడం పట్ల మేము చింతిస్తున్నాము. దయచేసి మీ రౌటర్‌ను రీస్టార్ట్ చేయండి, కేబుల్ కనెక్షన్లను తనిఖీ చేయండి మరియు మీ పరికరాన్ని మళ్లీ వై-ఫైకు కనెక్ట్ చేయండి. సమస్య కొనసాగితే మా సహాయ బృందాన్ని సంప్రదించండి.. Status: Awaiting customer confirmation.


In [16]:

# FINAL DATAFRAME COLUMN CHECK AND PREPARATION

# 1. Create actual language from original locale
language_mapping = {
    "en-US": "English",
    "hi-IN": "Hindi",
    "es-ES": "Spanish",
    "te-IN": "Telugu"
}

df["actual_language"] = df["locale"].map(language_mapping)

# 2. Check whether detected language is available
print("Detected language column exists:",
      "detected_language" in df.columns)

# 3. Create language correctness
df["language_correct"] = (
    df["actual_language"].str.lower()
    ==
    df["detected_language"].astype(str).str.lower()
)

# 4. Verify columns
print("\nAvailable columns:")
print(df.columns.tolist())

print("\nMissing values in actual language:",
      df["actual_language"].isna().sum())

print("\nLanguage detection results:")
print(df["language_correct"].value_counts())


Detected language column exists: True

Available columns:
['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments', 'language_name', 'detected_language', 'issue_category', 'actual_language', 'customer_reply', 'internal_english_log', 'language_correct']

Missing values in actual language: 0

Language detection results:
language_correct
True     42661
False     3395
Name: count, dtype: int64


In [17]:
# STEP 12: FINAL PROJECT RESULTS
final_results = df[[
    "utt",
    "actual_language",
    "detected_language",
    "language_correct",
    "issue_category",
    "customer_reply",
    "internal_english_log"
]].copy()
print("FINAL MULTILINGUAL SUPPORT DESK RESULTS")
display(final_results.head(15))
print("\nIssue category distribution:")
print(df["issue_category"].value_counts())
print("\nTotal processed messages:", len(final_results))

FINAL MULTILINGUAL SUPPORT DESK RESULTS


,utt,actual_language,detected_language,language_correct,issue_category,customer_reply,internal_english_log
0,wake me up at nine am on friday,English,English,True,Other Issue,Thank you for contacting support. Could you pl...,Customer language: English. Issue category: Ot...
1,set an alarm for two hours from now,English,English,True,Other Issue,Thank you for contacting support. Could you pl...,Customer language: English. Issue category: Ot...
2,olly quiet,English,Unknown,False,Other Issue,Thank you for contacting support. Could you pl...,Customer language: Unknown. Issue category: Ot...
3,stop,English,Unknown,False,Other Issue,Thank you for contacting support. Could you pl...,Customer language: Unknown. Issue category: Ot...
4,olly pause for ten seconds,English,English,True,Other Issue,Thank you for contacting support. Could you pl...,Customer language: English. Issue category: Ot...
5,pause for ten seconds,English,English,True,Other Issue,Thank you for contacting support. Could you pl...,Customer language: English. Issue category: Ot...
6,make the lighting bit more warm here,English,English,True,Other Issue,Thank you for contacting support. Could you pl...,Customer language: English. Issue category: Ot...
7,please set the lighting suitable for reading,English,English,True,Other Issue,Thank you for contacting support. Could you pl...,Customer language: English. Issue category: Ot...
8,time to sleep,English,English,True,Other Issue,Thank you for contacting support. Could you pl...,Customer language: English. Issue category: Ot...
9,time to sleep olly,English,Unknown,False,Other Issue,Thank you for contacting support. Could you pl...,Customer language: Unknown. Issue category: Ot...



Issue category distribution:
issue_category
Other Issue       45457
Network Issue       316
Billing Issue       250
Slow Service         27
Password Issue        6
Name: count, dtype: int64

Total processed messages: 46056


In [18]:

# STEP 13: TEST THE MULTILINGUAL SUPPORT DESK

test_messages = [
    # ENGLISH
    {
        "message": "My internet connection is not working. I cannot access any website.",
        "language": "English",
        "issue": "Network Issue"
    },
    {
        "message": "I forgot my password and cannot log into my account.",
        "language": "English",
        "issue": "Password Issue"
    },
    {
        "message": "I was charged twice for the same monthly bill.",
        "language": "English",
        "issue": "Billing Issue"
    },
    {
        "message": "My internet speed is very slow and videos keep buffering.",
        "language": "English",
        "issue": "Slow Service"
    },
    {
        "message": "I need help updating my account information.",
        "language": "English",
        "issue": "Other Issue"
    },

    # HINDI
    {
        "message": "मेरा इंटरनेट काम नहीं कर रहा है। मैं किसी भी वेबसाइट को नहीं खोल पा रहा हूँ।",
        "language": "Hindi",
        "issue": "Network Issue"
    },
    {
        "message": "मैं अपना पासवर्ड भूल गया हूँ और लॉगिन नहीं कर पा रहा हूँ।",
        "language": "Hindi",
        "issue": "Password Issue"
    },
    {
        "message": "मेरे बिल में गलत भुगतान दिखाया गया है।",
        "language": "Hindi",
        "issue": "Billing Issue"
    },
    {
        "message": "मेरा इंटरनेट बहुत धीमा चल रहा है।",
        "language": "Hindi",
        "issue": "Slow Service"
    },
    {
        "message": "मुझे अपने अकाउंट की जानकारी अपडेट करनी है।",
        "language": "Hindi",
        "issue": "Other Issue"
    },

    # SPANISH
    {
        "message": "Mi conexión a internet no funciona y no puedo abrir ninguna página web.",
        "language": "Spanish",
        "issue": "Network Issue"
    },
    {
        "message": "Olvidé mi contraseña y no puedo iniciar sesión en mi cuenta.",
        "language": "Spanish",
        "issue": "Password Issue"
    },
    {
        "message": "Me cobraron dos veces por la misma factura.",
        "language": "Spanish",
        "issue": "Billing Issue"
    },
    {
        "message": "Mi internet está muy lento y los videos se quedan cargando.",
        "language": "Spanish",
        "issue": "Slow Service"
    },
    {
        "message": "Necesito ayuda para actualizar la información de mi cuenta.",
        "language": "Spanish",
        "issue": "Other Issue"
    },

    # TELUGU
    {
        "message": "నా ఇంటర్నెట్ పనిచేయడం లేదు. ఏ వెబ్‌సైట్ కూడా తెరవడం లేదు.",
        "language": "Telugu",
        "issue": "Network Issue"
    },
    {
        "message": "నేను నా పాస్‌వర్డ్ మర్చిపోయాను. నా ఖాతాలో లాగిన్ కాలేకపోతున్నాను.",
        "language": "Telugu",
        "issue": "Password Issue"
    },
    {
        "message": "నా బిల్లులో తప్పు చెల్లింపు చూపిస్తోంది.",
        "language": "Telugu",
        "issue": "Billing Issue"
    },
    {
        "message": "నా ఇంటర్నెట్ చాలా నెమ్మదిగా పనిచేస్తోంది.",
        "language": "Telugu",
        "issue": "Slow Service"
    },
    {
        "message": "నా ఖాతా వివరాలను అప్డేట్ చేయడానికి సహాయం కావాలి.",
        "language": "Telugu",
        "issue": "Other Issue"
    }
]

test_results = []

for item in test_messages:

    message = item["message"]

    detected_language = detect_language(message)
    identified_issue = identify_issue(message)

    reply = generate_support_reply(
        identified_issue,
        detected_language
    )

    internal_log = (
        f"Customer language: {detected_language}. "
        f"Issue: {identified_issue}. "
        f"Original message: {message}. "
        f"Status: Awaiting customer confirmation."
    )

    test_results.append({
        "Customer Message": message,
        "Expected Language": item["language"],
        "Detected Language": detected_language,
        "Language Correct": detected_language == item["language"],
        "Expected Issue": item["issue"],
        "Identified Issue": identified_issue,
        "Issue Correct": identified_issue == item["issue"],
        "Customer Reply": reply,
        "English Internal Log": internal_log
    })

test_df = pd.DataFrame(test_results)

print("========== SUPPORT DESK TEST RESULTS ==========")

display(test_df)

print("\n========== TEST SUMMARY ==========")

print("Total test messages:", len(test_df))

print(
    "Correct language detections:",
    test_df["Language Correct"].sum(),
    "/",
    len(test_df)
)

print(
    "Correct issue identifications:",
    test_df["Issue Correct"].sum(),
    "/",
    len(test_df)
)

print(
    "Language accuracy:",
    round(test_df["Language Correct"].mean() * 100, 2),
    "%"
)

print(
    "Issue identification accuracy:",
    round(test_df["Issue Correct"].mean() * 100, 2),
    "%"
)


========== SUPPORT DESK TEST RESULTS ==========


,Customer Message,Expected Language,Detected Language,Language Correct,Expected Issue,Identified Issue,Issue Correct,Customer Reply,English Internal Log
0,My internet connection is not working. I canno...,English,English,True,Network Issue,Network Issue,True,We are sorry that your network is not working....,Customer language: English. Issue: Network Iss...
1,I forgot my password and cannot log into my ac...,English,English,True,Password Issue,Password Issue,True,We can help you with your login problem. Pleas...,Customer language: English. Issue: Password Is...
2,I was charged twice for the same monthly bill.,English,English,True,Billing Issue,Billing Issue,True,We can help you with your billing concern. Ple...,Customer language: English. Issue: Billing Iss...
3,My internet speed is very slow and videos keep...,English,Unknown,False,Slow Service,Network Issue,False,Thank you for contacting support. Could you pl...,Customer language: Unknown. Issue: Network Iss...
4,I need help updating my account information.,English,English,True,Other Issue,Other Issue,True,Thank you for contacting support. Could you pl...,Customer language: English. Issue: Other Issue...
5,मेरा इंटरनेट काम नहीं कर रहा है। मैं किसी भी व...,Hindi,Hindi,True,Network Issue,Network Issue,True,"आपका नेटवर्क काम नहीं कर रहा है, इसके लिए हमें...",Customer language: Hindi. Issue: Network Issue...
6,मैं अपना पासवर्ड भूल गया हूँ और लॉगिन नहीं कर ...,Hindi,Hindi,True,Password Issue,Password Issue,True,हम आपकी लॉगिन समस्या में मदद कर सकते हैं। कृपय...,Customer language: Hindi. Issue: Password Issu...
7,मेरे बिल में गलत भुगतान दिखाया गया है।,Hindi,Hindi,True,Billing Issue,Billing Issue,True,हम आपकी बिलिंग समस्या में मदद कर सकते हैं। कृप...,Customer language: Hindi. Issue: Billing Issue...
8,मेरा इंटरनेट बहुत धीमा चल रहा है।,Hindi,Hindi,True,Slow Service,Network Issue,False,"आपका नेटवर्क काम नहीं कर रहा है, इसके लिए हमें...",Customer language: Hindi. Issue: Network Issue...
9,मुझे अपने अकाउंट की जानकारी अपडेट करनी है।,Hindi,Hindi,True,Other Issue,Other Issue,True,सहायता टीम से संपर्क करने के लिए धन्यवाद। कृपय...,Customer language: Hindi. Issue: Other Issue. ...



========== TEST SUMMARY ==========
Total test messages: 20
Correct language detections: 19 / 20
Correct issue identifications: 16 / 20
Language accuracy: 95.0 %
Issue identification accuracy: 80.0 %


In [19]:
#Test one customer at a time
# STEP 14: INTERACTIVE CUSTOMER SUPPORT TEST
customer_message = input("Enter your customer problem: ")
support_desk(customer_message)

Enter your customer problem: My Wi-Fi is not connecting to my laptop.
CUSTOMER MESSAGE:
My Wi-Fi is not connecting to my laptop.

DETECTED LANGUAGE:
English

IDENTIFIED ISSUE:
Network Issue

CUSTOMER SUPPORT REPLY:
We are sorry that your network is not working. Please restart your router, check the cable connections, and reconnect your device to Wi-Fi. If the problem continues, please contact our support team.

INTERNAL LOG (ENGLISH):
Customer language: English. Issue: Network Issue. Original message: My Wi-Fi is not connecting to my laptop.. Recommended response: We are sorry that your network is not working. Please restart your router, check the cable connections, and reconnect your device to Wi-Fi. If the problem continues, please contact our support team.. Status: Awaiting customer confirmation.


In [20]:

# STEP 15: MIXED LANGUAGE DETECTION


import re
from langdetect import detect_langs

def detect_languages_in_message(message):

    languages_found = []

    # Detect Hindi script
    if re.search(r'[\u0900-\u097F]', message):
        languages_found.append("Hindi")

    # Detect Telugu script
    if re.search(r'[\u0C00-\u0C7F]', message):
        languages_found.append("Telugu")

    # Remove Hindi and Telugu characters before checking
    # the remaining Latin-script text
    latin_text = re.sub(
        r'[\u0900-\u097F\u0C00-\u0C7F]',
        ' ',
        message
    )

    # Detect English or Spanish in the remaining text
    if len(latin_text.split()) >= 4:
        try:
            predictions = detect_langs(latin_text)

            best = predictions[0]

            if best.prob >= 0.70:
                if best.lang == "en":
                    languages_found.append("English")
                elif best.lang == "es":
                    languages_found.append("Spanish")

        except:
            pass

    # Remove duplicate language names
    languages_found = list(dict.fromkeys(languages_found))

    if not languages_found:
        return ["Unknown"]

    return languages_found


In [21]:

# STEP 16: TEST MIXED LANGUAGE MESSAGES
#Ask the customer to choose a language


def get_preferred_language(detected_languages):

    if len(detected_languages) > 1:

        language_options = " or ".join(detected_languages)

        print(
            f"\nYour message contains "
            f"{language_options}."
        )

        print(
            f"Would you prefer a response in "
            f"{language_options}?"
        )

        while True:
            choice = input(
                "Enter your preferred language: "
            ).strip().lower()

            for language in detected_languages:
                if choice == language.lower():
                    return language

            print("Please enter one of the detected languages.")

    elif detected_languages[0] != "Unknown":

        return detected_languages[0]

    else:

        print("We could not identify the language.")
        return input(
            "Please enter English, Hindi, Spanish or Telugu: "
        ).strip().title()


In [22]:
#Connect language selection with the support desk

support_logs = []

def support_desk(message):

    print("\n========== SUPPORT DESK ==========")

    # Detect languages
    detected_languages = detect_languages_in_message(message)

    print("Detected languages:", detected_languages)

    # Ask customer to choose if message is mixed
    preferred_language = get_preferred_language(
        detected_languages
    )

    #Identify customer problem
    issue = identify_issue(message)

    #Generate reply in selected language
    reply = generate_support_reply(
        issue,
        preferred_language
    )

    print("\nCustomer message:", message)
    print("Identified issue:", issue)
    print("Reply language:", preferred_language)

    print("\nSupport desk response:")
    print(reply)

    # 5. Save internal log in English
    support_logs.append({
        "customer_message": message,
        "detected_languages": ", ".join(detected_languages),
        "preferred_language": preferred_language,
        "identified_issue": issue,
        "support_response": reply,
        "status": "Processed"
    })

    print("\nInternal log saved in English.")
    print("==================================")


In [23]:
#Test the new functionality

#support_desk("My internet काम नहीं कर रहा")
#support_desk("My network పనిచేయడం లేదు")
support_desk("My password is not working")


========== SUPPORT DESK ==========
Detected languages: ['Unknown']
We could not identify the language.
Please enter English, Hindi, Spanish or Telugu: My Wi-Fi is not connecting to my laptop.

Customer message: My password is not working
Identified issue: Password Issue
Reply language: My Wi-Fi Is Not Connecting To My Laptop.

Support desk response:
Thank you for contacting support. Could you please explain your problem in more detail? Our support team will help you.

Internal log saved in English.


In [35]:
#My Wi-Fi is not connecting to my laptop.

#I cannot reset my account password.

#I paid my bill but the payment is not showing.

#My password is not working

#I have a billing problem

#मेरा पासवर्ड काम नहीं कर रहा

#My internet is extremely slow today.

#मेरा वाईफाई कनेक्ट नहीं हो रहा है।

#मेरा बिल गलत है

#मैं अपना पासवर्ड रीसेट नहीं कर पा रहा हूँ।

#मेरा इंटरनेट काम नहीं कर रहा है

#నా వైఫై కనెక్ట్ కావడం లేదు.

#నా పాస్‌వర్డ్ మర్చిపోయాను.

In [28]:
#Flask installation for web page
!pip install -q flask
print("Installation completed")

Installation completed


In [37]:
#Create the Flask application
from flask import Flask, request, render_template_string
from threading import Thread
app = Flask(__name__)
# Supported customer reply languages
SUPPORTED_LANGUAGES = [
    "English", "Hindi", "Spanish", "Telugu"
]
# Webpage design
HTML_PAGE = """
<!DOCTYPE html>
<html>
<head>
    <title>Multilingual Support Desk</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            background: #f2f5f9;
            margin: 0;
            padding: 30px;
        }
        .container {
            max-width: 650px;
            margin: auto;
            background: white;
            padding: 30px;
            border-radius: 12px;
            box-shadow: 0 3px 12px #cccccc;
        }
        h1 {
            color: #174a7e;
            text-align: center;
        }
        p {
            color: #444;
        }
        textarea {
            width: 100%;
            height: 120px;
            padding: 12px;
            box-sizing: border-box;
            border: 1px solid #bbb;
            border-radius: 6px;
            font-size: 16px;
        }
        select {
            width: 100%;
            padding: 12px;
            margin-top: 10px;
            border-radius: 6px;
        }
        button {
            background: #174a7e;
            color: white;
            padding: 12px 22px;
            border: none;
            border-radius: 6px;
            cursor: pointer;
            margin-top: 15px;
            font-size: 16px;
        }
        .result {
            background: #eaf4ff;
            padding: 18px;
            margin-top: 20px;
            border-radius: 8px;
        }
        .reply {
            background: #e8f7ec;
            padding: 18px;
            margin-top: 15px;
            border-radius: 8px;
        }
        .error {
            color: #b00020;
        }
    </style>
</head>
<body>
<div class="container">
    <h1>Multilingual Support Desk</h1>
    <p>
        Welcome! Describe your problem below.
        Our support desk can assist in English,
        Hindi, Spanish and Telugu.
    </p>
    <form method="POST">
        <label><b>Enter your message:</b></label>
        <textarea name="message"
            placeholder="Example: My internet is not working"
            required>{{ message }}</textarea>
        {% if show_language_choice %}
            <div class="result">
                <h3>Language preference</h3>
                <p>{{ language_question }}</p>
                <label><b>Choose your preferred language:</b></label>
                <select name="preferred_language" required>
                    {% for language in language_options %}
                    <option value="{{ language }}">
                        {{ language }}
                    </option>
                    {% endfor %}
                </select>
            </div>
        {% endif %}
        <button type="submit">Submit Message</button>
    </form>
    {% if error %}
        <p class="error">{{ error }}</p>
    {% endif %}
    {% if reply %}
        <div class="result">
            <h3>Support Desk Analysis</h3>
            <p><b>Detected languages:</b> {{ detected }}</p>
            <p><b>Identified issue:</b> {{ issue }}</p>
            <p><b>Reply language:</b> {{ selected_language }}</p>
        </div>
        <div class="reply">
            <h3>Our Support Response</h3>
            <p>{{ reply }}</p>
        </div>
    {% endif %}
</div>
</body>
</html>
"""
@app.route("/", methods=["GET", "POST"])
def home():
    message = ""
    detected = []
    issue = ""
    reply = ""
    selected_language = ""
    error = ""
    show_language_choice = False
    language_options = []
    language_question = ""
    if request.method == "POST":
        message = request.form.get("message", "").strip()
        preferred_language = request.form.get(
            "preferred_language", ""
        ).strip()
        if not message:
            error = "Please enter a customer message."
        else:
            # 1. Detect languages
            detected = detect_languages_in_message(message)
            # 2. Ask preference for mixed or unknown language
            if len(detected) > 1 or detected[0] == "Unknown":
                language_options = (
                    detected if detected[0] != "Unknown"
                    else SUPPORTED_LANGUAGES
                )
                if preferred_language not in language_options:
                    show_language_choice = True
                    if detected[0] == "Unknown":
                        language_question = (
                            "We could not confidently identify "
                            "your language. Please choose one."
                        )
                    else:
                        language_question = (
                            "Your message contains "
                            + " and ".join(detected)
                            + ". Would you prefer a response in "
                            + " or ".join(detected)
                            + "?"
                        )
                else:
                    selected_language = preferred_language

            else:
                # Single-language message
                selected_language = detected[0]
            # 3. Generate the reply after language selection
            if selected_language:
                issue = identify_issue(message)
                reply = generate_support_reply(
                    issue,
                    selected_language
                )
                # 4. Save English internal log
                support_logs.append({
                    "customer_issue_summary":
                        f"Customer reported a {issue.lower()} issue.",
                    "detected_languages":
                        ", ".join(detected),
                    "preferred_language":
                        selected_language,
                    "identified_issue":
                        issue,
                    "internal_status":
                        "Processed"
                })
    return render_template_string(
        HTML_PAGE,
        message=message,
        detected=", ".join(detected),
        issue=issue,
        reply=reply,
        selected_language=selected_language,
        error=error,
        show_language_choice=show_language_choice,
        language_options=language_options,
        language_question=language_question
    )
print("Flask application created successfully!")

Flask application created successfully!


In [38]:
#Step 26 – Start the Flask website

from google.colab import output

def run_flask():
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )

flask_thread = Thread(target=run_flask)
flask_thread.daemon = True
flask_thread.start()

print("Opening the support desk website...")

output.serve_kernel_port_as_iframe(
    5000,
    height=750
)


Opening the support desk website...
 * Serving Flask app '__main__'


<IPython.core.display.Javascript object>

 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


In [34]:
#Check the internal logs

logs_df = pd.DataFrame(support_logs)

display(logs_df)

logs_df.to_csv(
    "final_support_desk_logs.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Internal logs exported successfully!")


,customer_message,detected_languages,preferred_language,identified_issue,support_response,status,customer_issue_summary,internal_status
0,My password is not working,Unknown,My Wi-Fi Is Not Connecting To My Laptop.,Password Issue,Thank you for contacting support. Could you pl...,Processed,NaN,NaN
1,NaN,Unknown,English,Network Issue,NaN,NaN,Customer reported a network issue issue.,Processed
2,NaN,Unknown,English,Billing Issue,NaN,NaN,Customer reported a billing issue issue.,Processed


Internal logs exported successfully!
